<a href="https://colab.research.google.com/github/Lattemelia/Portfolio/blob/main/%E3%80%900624%EF%BD%9E%E7%8F%BE%E5%9C%A8%E4%BD%9C%E6%88%90%E4%B8%AD%E3%80%91/%E4%B8%AD%E5%8F%A4%E8%BB%8A%E3%81%AE%E4%BE%A1%E6%A0%BC%E4%BA%88%E6%B8%AC(KaggleConpe).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [123]:
#デ－タの読み込み
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [124]:
#ツールの読み込み
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import os


In [125]:
#データをデータフレームに落とし込み
PATH = r"/content/drive/MyDrive/Colab Notebooks"

test_df = pd.read_csv(PATH + r"/test.csv")
train_df = pd.read_csv(PATH + r"/train.csv")

marged_df = pd.concat([test_df,train_df],axis=0)

In [126]:
#名称をカテゴリ変数に変換
cols_to_lower = ['brand', 'model', 'engine', 'fuel_type', 'transmission']

for col in cols_to_lower:
    marged_df[col] = marged_df[col].str.lower().str.strip()

In [127]:
#在庫数がわずかなブランドの削除
brands_to_exclude = ['smart', 'polestar']
rows_to_drop = marged_df[marged_df['brand'].isin(brands_to_exclude)].index
marged_df = marged_df.drop(rows_to_drop)

In [128]:
#エンジンの情報に燃料情報が含まれていれば燃料の欠損値補完
import re

fuel_pattern = r'(gasoline|diesel|e85 flex fuel|flex fuel|electric|hybrid|plug-in hybrid)'

for df in [marged_df]:
    df['fuel_type'] = df['fuel_type'].replace(['–', 'not supported'], np.nan)
    extracted = df['engine'].str.extract(fuel_pattern, expand=False, flags=re.IGNORECASE)
    df['fuel_type'] = df['fuel_type'].fillna(extracted)

for df in [marged_df]:
    df['fuel_type'] = df['fuel_type'].replace('e85 flex fuel', 'flex fuel')

In [129]:
#clean_titleをカテゴリ変数に変換
marged_df['clean_title'] = (marged_df['clean_title'] == 'Yes').astype(int)

#外装色・内装色などの情報を削除
marged_df = marged_df.drop(columns=['ext_col', 'int_col'])

#製造年数から経年数を算出
marged_df['age'] = 2026 - marged_df['model_year']

In [130]:
#エンジンの情報から馬力・排気量・気筒数を特徴量として作成
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

for df in [marged_df]:
    df['hp'] = df['engine'].str.extract(r'(\d+\.?\d*)hp', flags=re.IGNORECASE).astype(float)
    df['liters'] = df['engine'].str.extract(r'(\d+\.?\d*)l', flags=re.IGNORECASE).astype(float)
    df['cylinder'] = df['engine'].str.extract(r'(\d+\.?\d*) cylinder', flags=re.IGNORECASE).astype(float)

#HPに欠損値があればほかの特徴量をもとに推定
cols_to_impute = ['hp', 'liters', 'cylinder', 'milage', 'age']

ii_imputer = IterativeImputer(
    max_iter=10,
    random_state=42,
    initial_strategy='median'
)

marged_df[cols_to_impute] = ii_imputer.fit_transform(marged_df[cols_to_impute])

In [131]:
import pandas as pd
import numpy as np

# -----------------------------
# 1. 小文字化
# -----------------------------
marged_df["model"] = marged_df["model"].str.lower()
marged_df["brand"] = marged_df["brand"].str.lower()

# -----------------------------
# 2. 1～7単語目を分割
# -----------------------------
words = marged_df["model"].str.split(expand=True)

for i in range(7):
    if i in words.columns:
        marged_df[f"word{i+1}"] = words[i]
    else:
        marged_df[f"word{i+1}"] = np.nan

In [132]:
import pandas as pd


def extract_pure_model(row):
    # brandを小文字に統一（比較のブレを防ぐ）
    brand = (
        str(row["brand"]).lower().strip() if pd.notna(row["brand"]) else ""
    )

    w1 = row["word1"]
    w2 = row["word2"]
    w3 = row["word3"]
    w4 = row["word4"]

    # 欠損対策＋小文字に統一
    w1 = "" if pd.isna(w1) else str(w1).lower().strip()
    w2 = "" if pd.isna(w2) else str(w2).lower().strip()
    w3 = "" if pd.isna(w3) else str(w3).lower().strip()
    w4 = "" if pd.isna(w4) else str(w4).lower().strip()

    # ─── ① Rover (Land Rover) ───
    if w1 == "rover":
        if w2 == "range":
            if w4 in ["velar", "evoque", "sport"]:
                return w2 + w3 + w4
            else:
                return w2 + w3
        else:
            return w2

    # ─── ② 元のコードで Rover ブロックの外にあった Range 判定（安全のため統合） ───
    # 1単語目が Rover ではなく、2単語目が Range から始まるイレギュラー対策
    if w2 == "range":
        if w4 in ["velar", "evoque", "sport"]:
            return w2 + w3 + w4
        else:
            return w2 + w3

    # ─── ③ Mercedes ───
    if w1 in ["c-class", "cls-class"]:
        return w2 + w3

    if w1 == "amg":
        return w2

    if w1 == "alpina":
        return w2

    # ─── ④ Audi RS ───
    if brand == "audi" and w1 == "rs":
        return w1 + w2

    # ─── ⑤ Grand Cherokee ───
    if w1 == "grand" and w2 == "cherokee":
        return w1 + w2

    # ─── ⑥ Continental GT ───
    if w1 == "continental" and w2 == "gt":
        return w1 + w2

    # ─── ⑦ Tesla Model（Audi model 仕様書反映） ───
    if (brand == "tesla" and w1 == "model") or (
        brand == "audi" and w1 == "model"
    ):
        return w1 + w2

    # ─── ⑧ 2単語目が無い ───
    if w2 == "":
        return w1

    # ─── ⑨ その他 ───
    return w1


# 🚀 実行コード
marged_df["pure_model"] = marged_df.apply(extract_pure_model, axis=1)

In [133]:
#説明変数(X)と目的変数(y)に分ける
X, y = marged_df.drop('price', axis=1), marged_df['price']

cat_cols = X.select_dtypes(include=['object']).columns
num_cols = X.select_dtypes(exclude=['object']).columns

median = X[num_cols].median()

X[num_cols] = X[num_cols].fillna(median)
marged_df[num_cols] = marged_df[num_cols].fillna(median)

In [134]:
y = np.log1p(y)

In [145]:
marged_df.isnull().sum()

,0
id,0
brand,0
model_year,0
milage,0
fuel_type,1588
engine,6
transmission,0
accident,4084
clean_title,0
price,125685


In [136]:
global_mean = y.mean()

for col in cat_cols:
    temp = pd.concat([X[col], y], axis=1)
    means = temp.groupby(col)['price'].mean()

    X[col] = X[col].map(means)
    marged_df[col] = marged_df[col].map(means)

In [137]:
features_to_log = ['milage', 'hp']

for col in features_to_log:
    X[col] = np.log1p(X[col])
    marged_df[col] = np.log1p(marged_df[col])

In [138]:
drop_col = ['word1','word2','word3','word4','word5','word6','word7','model']

marged_df = marged_df.drop(columns=drop_col)

In [139]:
import pandas as pd

# ─── 1. price が入っている（欠損していない）行を Train にする ───
train_df = marged_df[marged_df["price"].notnull()].copy()

# ─── 2. price が欠損している行を Test にする ───
test_df = marged_df[marged_df["price"].isnull()].copy()

# ─── 3. Testデータから予測対象である 'price' 列を落とす（お作法） ───
test_df = test_df.drop(columns=["price"])

# 📊 分割後のデータ件数を確認
print(f"📁 元の全体のデータ数: {len(marged_df)}")
print(f"🚗 Train（学習用）のデータ数: {len(train_df)}")
print(f"🔮 Test（予測用）のデータ数 : {len(test_df)}")

📁 元の全体のデータ数: 314208
🚗 Train（学習用）のデータ数: 188523
🔮 Test（予測用）のデータ数 : 125685


In [140]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [141]:
model = lgb.LGBMRegressor(
    max_depth=10,
    learning_rate=0.008,
    n_estimators=8000,
    min_child_samples=50,
    random_state=42,
    importance_type='gain',
    verbosity=-1
)

In [142]:
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='mae',
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=250)
    ]
)

Training until validation scores don't improve for 50 rounds
[250]	valid_0's l1: 0.713887	valid_0's l2: 0.685629
[500]	valid_0's l1: 0.267375	valid_0's l2: 0.223657
[750]	valid_0's l1: 0.225772	valid_0's l2: 0.213804
[1000]	valid_0's l1: 0.220536	valid_0's l2: 0.213086
[1250]	valid_0's l1: 0.220164	valid_0's l2: 0.212807
Early stopping, best iteration is:
[1392]	valid_0's l1: 0.220095	valid_0's l2: 0.212707


LGBMRegressor(importance_type='gain', learning_rate=0.008, max_depth=10,
              min_child_samples=50, n_estimators=8000, random_state=42,
              verbosity=-1)

In [144]:
print("y_val NaN:", np.isnan(y_val).sum())
print("val_preds NaN:", np.isnan(val_preds).sum())

y_val NaN: 25129
val_preds NaN: 0


In [146]:
val_preds = np.expm1(model.predict(X_val))
mae = mean_absolute_error(np.expm1(y_val), val_preds)
print(f"MAE: {mae:,.0f}")

ValueError: Input contains NaN.